<a href="https://colab.research.google.com/github/SsemuliJoseph/Sunbird/blob/main/Luganda_NLP_Pipeline_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# English → Luganda NLP Pipeline
### A complete practical walkthrough of the Hugging Face ecosystem (Chapters 1–11)

**Author:** Joseph Ssemuli Sunbird AI

This notebook is built around one real engineering question: *can we teach a model to translate English into Luganda, starting from raw, messy data, and end with something a real user could actually open in a browser?*

That single question forces you to touch almost every layer of the modern NLP stack — which is exactly why it's a good capstone before an internship at an organization like Sunbird AI, where you'll be doing this for real, on real Ugandan languages.

#### How an NLP engineer actually thinks about a project like this

Before any code, a working engineer asks four questions, in this order:

1. **What data do I have, and how good is it?** (Phase 1)
2. **What is the simplest model that could plausibly solve this?** Build that first, time-box it, and use it to set a floor you must beat. (Phase 2)
3. **Given the floor I just set, is a bigger/fancier approach worth the extra cost?** (Phase 3–4)
4. **How do I make this resilient and usable by someone who isn't me?** (Phase 5–7)

This ordering is not arbitrary — it's the same reason hospitals triage before they operate. Jumping straight to the biggest model (an 8-billion-parameter LLM) without first knowing your data quality or having a baseline number to compare against is how engineers burn a week of GPU time and learn nothing. We will not do that here.

#### The shape of the project

| Phase | What we build | HF Course chapters it rehearses | Core engineering question |
|---|---|---|---|
| 0 | Environment + sanity checks | 1–2 | "Is my toolchain even working?" |
| 1 | Curated dataset from Argilla | 3, 5 | "How good is my data, really?" |
| 2 | Seq2seq baseline (NLLB) | 3, 7 | "What's the simplest thing that works?" |
| 3 | Tokenizer internals + Hub repo | 4, 6 | "What is actually happening to my text, and how do I share results?" |
| 4 | QLoRA fine-tune (Qwen2.5) | 11 | "Is a bigger model worth the extra compute?" |
| 5 | Debugging toolkit | 8 | "What do I do when it breaks at 2am before a demo?" |
| 6 | Gradio demo + Spaces | 9 | "Can someone who isn't me actually use this?" |
| 7 | Model card + review | 10 | "Is this reproducible and honestly documented?" |

#### One-time setup
Runtime → Change runtime type → **T4 GPU**. Then open the key icon in the left sidebar and add two Colab secrets: `HF_TOKEN` (a Hugging Face token with write access — generate one at huggingface.co/settings/tokens) and `ARGILLA_API_KEY` (only needed if you re-pull annotations in Phase 1). Toggle both secrets to "Notebook access: on."


---
## Phase 0 — Environment Setup & the `pipeline()` Abstraction
### *HF Course Chapters 1–2*

#### The engineering principle: pin your dependencies, every time

Every library in the Hugging Face ecosystem — `transformers`, `trl`, `peft`, `accelerate`, `bitsandbytes` — is developed somewhat independently, by different teams, releasing on different schedules. They are designed to interoperate, but only for ranges of versions that have actually been tested together. `trl.SFTTrainer`, for instance, has changed its constructor signature more than once as the library matured; `BitsAndBytesConfig` defaults have shifted; `Seq2SeqTrainingArguments` renamed `evaluation_strategy` to `eval_strategy` at one point.

If you run a bare `pip install transformers` today and again in three months, you can get two different dependency trees, and one of them might simply not work together. This is not a Hugging Face-specific problem — it's the universal cost of building on a fast-moving open-source stack. The fix is always the same: **pin version ranges that you have verified work together, and only widen them deliberately.** That's what the cell below does. Treat it as a habit, not a one-off — you'll want this discipline at Sunbird AI too, where reproducibility across teammates' machines matters.


In [ ]:
# Pinned, mutually-compatible stack (verified working together as of mid-2026).
# If a future Colab image already has compatible versions, this just confirms them.
!pip install -q -U \
    "transformers>=4.55,<4.57" \
    "datasets>=2.20,<3.1" \
    "accelerate>=0.34,<1.2" \
    "evaluate>=0.4.2" \
    "sacrebleu>=2.4" \
    "sentencepiece>=0.2.0" \
    "peft>=0.13,<0.16" \
    "trl>=0.12,<0.16" \
    "bitsandbytes>=0.43.1" \
    "gradio>=4.44" \
    "huggingface_hub>=0.25"

print("Install complete. Restart not required for this version set.")


Install complete. Restart not required for this version set.


Next, let's verify the installed versions and ensure GPU is available.

In [ ]:
import torch, transformers, datasets, accelerate, peft, trl
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"datasets     : {datasets.__version()}")
print(f"accelerate   : {accelerate.__version()}")
print(f"peft         : {peft.__version()}")
print(f"trl          : {trl.__version()}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU.")

torch        : 2.11.0+cu128
transformers : 4.56.2


AttributeError: module 'datasets' has no attribute '__version'

Now, let's log in to Hugging Face Hub using your `HF_TOKEN`.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print("Logged in to Hugging Face Hub.")

Logged in to Hugging Face Hub.


Let's quickly confirm the token is loaded correctly.

In [ ]:
print(f"HF_TOKEN (first 4 chars): {HF_TOKEN[:4]}")

HF_TOKEN (first 4 chars): hf_I


Next, we'll set up the data for the baseline model, which includes converting the golden dataset and loading the full corpus.

In [ ]:
import pandas as pd
import ast
from datasets import Dataset, load_dataset

def extract_golden_dataset(csv_path: str) -> pd.DataFrame:
    """Turn raw Argilla export into clean (english, luganda, source) pairs."""
    df = pd.read_csv(csv_path)
    clean_pairs = []

    for _, row in df.iterrows():
        if pd.isna(row.get('quality.responses')):
            continue
        try:
            quality_list = ast.literal_eval(row['quality.responses'])
            quality = quality_list[0] if quality_list else "No"
        except (ValueError, SyntaxError):
            continue

        english_text = row['english']

        if quality == "Yes":
            clean_pairs.append({
                "english": english_text,
                "luganda": row['luganda'],
                "source": "human_verified",
            })
        elif quality == "Needs Editing" and pd.notna(row.get('correction.responses')):
            try:
                correction_list = ast.literal_eval(row['correction.responses'])
                luganda_text = correction_list[0] if correction_list else row['luganda']
                clean_pairs.append({
                    "english": english_text,
                    "luganda": luganda_text,
                    "source": "human_corrected",
                })
            except (ValueError, SyntaxError):
                continue
        # quality == "No" -> drop (unsalvageable)

    return pd.DataFrame(clean_pairs)

golden_df = extract_golden_dataset("argilla_luganda_backup.csv")
print(f"Golden Dataset Size: {len(golden_df)} rows")
golden_df.head()

# Convert the golden eval set into a HF Dataset object (Ch.3: Datasets library).
golden_eval_dataset = Dataset.from_pandas(golden_df[["english", "luganda"]], preserve_index=False)
print(golden_eval_dataset)

# Pull the larger public corpus for training.
print("\nDownloading public training corpus...")
full_corpus = load_dataset("kambale/luganda-english-parallel-corpus", split="train")
print(full_corpus)


Golden Dataset Size: 7 rows
Dataset({
    features: ['english', 'luganda'],
    num_rows: 7
})

Dataset({
    features: ['english', 'luganda'],
    num_rows: 50012
})


We will now set up the training and evaluation datasets for the NLLB baseline model.

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

MODEL_CHECKPOINT = "facebook/nllb-200-distilled-600M"
SOURCE_LANG = "eng_Latn"
TARGET_LANG = "lug_Latn"
MAX_LENGTH = 128
MAX_TRAIN_ROWS = 3000   # cap so this stays fast on a free T4
MAX_EVAL_ROWS = 300

# Reuse full_corpus from Phase 1 if present, else reload.
try:
    full_corpus
except NameError:
    full_corpus = load_dataset("kambale/luganda-english-parallel-corpus", split="train")

n = len(full_corpus)
train_n = min(int(n * 0.9), MAX_TRAIN_ROWS)
eval_n = min(int(n * 0.1), MAX_EVAL_ROWS)

split = full_corpus.shuffle(seed=42).select(range(train_n + eval_n)).train_test_split(
    test_size=eval_n, seed=42
)
train_dataset = split["train"]
eval_dataset = split["test"]
print(f"Train rows: {len(train_dataset)} | Eval rows: {len(eval_dataset)}")

Using device: cuda
Train rows: 3000 | Eval rows: 300


Ensure `evaluate` is installed before proceeding to training.

In [ ]:
# Ensure 'evaluate' is installed.
!pip install -q evaluate

Now, we will tokenize the datasets, load the model, set up metrics, and train the baseline translation model.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT, src_lang=SOURCE_LANG, tgt_lang=TARGET_LANG
)

def preprocess_function(examples):
    return tokenizer(
        examples["english"],
        text_target=examples["luganda"],
        max_length=MAX_LENGTH,
        truncation=True,
    )

print("Tokenizing...")
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_eval = eval_dataset.map(preprocess_function, batched=True, remove_columns=eval_dataset.column_names)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT).to(device)

metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [p.strip() for p in preds]
    labels = [[l.strip()] for l in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb-luganda-baseline",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=2,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting baseline training...")
trainer.train()
baseline_metrics = trainer.evaluate()
print("Baseline evaluation:", baseline_metrics)


Tokenizing...


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Starting baseline training...


TypeError: Accelerator.unwrap_model() got an unexpected keyword argument 'keep_torch_compile'

After training, we'll sanity-check the baseline model against the human-verified golden set.

In [ ]:
# Sanity-check the baseline against your 7-row human-verified gold set.
def translate_batch(texts, model, tokenizer, max_length=128):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
    forced_bos = tokenizer.convert_tokens_to_ids(TARGET_LANG)
    outputs = model.generate(**inputs, forced_bos_token_id=forced_bos, max_length=max_length)
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

sample_en = golden_eval_dataset["english"][:5]
sample_lg = golden_eval_dataset["luganda"][:5]
preds = translate_batch(sample_en, model, tokenizer)

for en, ref, pred in zip(sample_en, sample_lg, preds):
    print(f"EN  : {en}")
    print(f"REF : {ref}")
    print(f"PRED: {pred}")
    print("---")

EN  : Where can we get good resistant varieties that are highly marketable?
REF : Tusobola kuggya wa ebika ebirungi ebigumira embeera yonna bya ttunzi nnyo?
PRED: Wa we tuyinza okufuna ebika ebiruma ennyo ebiruma ennyo?
---
EN  : Every Monday  farmers are given advice on coffee growing.
REF : Buli Mande abalimi bawabulwa ku nnima y'emmwanyi
PRED: Buli Lunaku abalimi baweebwa amagezi ku kukuza akawa.
---
EN  : Farmers are given a platform to ask any agricultural related questions.
REF : Abalimi baweebwa omwagaanya okubuuza ekibuuzo kyonna ekikwata ku byobulimi n'obulunzi.
PRED: Abalimi baweebwa ekifo eky'okubuuza ebibuuzo byonna ebikwata ku by'obulimi.
---
EN  : Farming programs aired on radio help farmers to know the good planting seasons.
REF : Pulogulaamu z'ebyobulimi eziweerezebwa ku laadiyo ziyamba balimi okumanya sizoni ennungi ez'okusimbiramu.
PRED: Empapula z'ebyobulimi ezaakwatanga ku leediyo ziyamba abalimi okumanya ebiseera ebirungi eby'okusiga.
---
EN  : Maize leaf rust caus

Now, let's free up the GPU memory by unloading the baseline model.

In [ ]:
# Free VRAM before moving to Phase 3/4 — this is the single most common
# fix for the CUDA OOM errors you'll hit later if you skip it.
import gc
del model, trainer
gc.collect()
torch.cuda.empty_cache()
print("Baseline model unloaded, VRAM freed.")

Baseline model unloaded, VRAM freed.


Finally, let's try pushing the model to the Hugging Face Hub again. Since you've logged in and verified your token, this should work now.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import os

MODEL_CHECKPOINT = "facebook/nllb-200-distilled-600M"
HF_USERNAME = "ssemulijoseph"
REPO_NAME = f"{HF_USERNAME}/nllb-luganda-baseline"

# Find the actual checkpoint directory inside the output folder
output_dir = "./nllb-luganda-baseline"
checkpoints = [os.path.join(output_dir, d) for d in os.listdir(output_dir) if d.startswith("checkpoint")]
load_path = max(checkpoints, key=os.path.getmtime) if checkpoints else output_dir

print(f"Reloading model from: {load_path}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = AutoModelForSeq2SeqLM.from_pretrained(load_path)

# Pushing to Hugging Face Hub
# IMPORTANT: Only uncomment these after updating your HF_TOKEN to 'Write' access in Secrets
print(f"Target Hub repo: {REPO_NAME}")
tokenizer.push_to_hub(REPO_NAME)
model.push_to_hub(REPO_NAME)


Reloading model from: ./nllb-luganda-baseline/checkpoint-376
Target Hub repo: ssemulijoseph/nllb-luganda-baseline


HfHubHTTPError: (Request ID: Root=1-6a3109bd-5a23753a742d716005d7a6e7;8fdd797f-76e2-4008-a589-cdd02de902d3)

403 Forbidden: You don't have the rights to create a model under the namespace "ssemulijoseph".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.

The cell above prints the resolved versions. Get into the habit of checking this output rather than assuming the install "just worked" — a silent version mismatch is much harder to debug three cells later than catching it here, right after install.

In [ ]:
import torch, transformers, datasets, accelerate, peft, trl
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"accelerate   : {accelerate.__version__}")
print(f"peft         : {peft.__version__}")
print(f"trl          : {trl.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU.")


torch        : 2.11.0+cu128
transformers : 4.56.2
datasets     : 3.0.2
accelerate   : 1.1.1
peft         : 0.15.2
trl          : 0.15.2
CUDA available: True
GPU: Tesla T4


#### Why authenticate before you do anything else

Several things later in this notebook need a Hugging Face token: downloading certain datasets, and pushing your trained models/adapters back to the Hub. Logging in now, in Phase 0, means you find out immediately if your token is missing or malformed — not forty minutes into a training run when you try to push and it fails.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print("Logged in to Hugging Face Hub.")

Logged in to Hugging Face Hub.


In [ ]:
print(f"HF_TOKEN (first 4 chars): {HF_TOKEN[:4]}")

HF_TOKEN (first 4 chars): hf_I


#### `pipeline()` — the abstraction you'll use 80% of the time in industry

The HF course opens with `pipeline()` for a reason: in real work, most NLP tasks (sentiment, NER, summarization, translation between well-resourced language pairs) already have a community-trained model that does the job. `pipeline()` hides three things you'd otherwise wire up by hand:

1. Loading the correct tokenizer for the model
2. Running the tokenizer → model → detokenizer chain
3. Formatting the output into something readable

The reason we *can't* just call `pipeline("translation", model=...)` for English→Luganda and stop here is the reason this whole project exists: there is no mature, off-the-shelf Luganda translation pipeline on the Hub yet. That gap is exactly the kind of gap Sunbird AI works to close for Ugandan languages. We confirm the abstraction itself works correctly on a well-resourced pair (English→French) first — isolating "is my code right" from "is Luganda support good enough" before we go further.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Define the model checkpoint
MODEL_CHECKPOINT = "Helsinki-NLP/opus-mt-en-fr"

# Load tokenizer and model directly, bypassing the pipeline task recognition issue
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT).to(device)

# Prepare input text
text = "Data is the new oil, but only if it is clean."
input_ids = tokenizer(text, return_tensors="pt", padding=True).to(device).input_ids

# Generate translation
# For MarianMT models, the tokenizer automatically handles the target language tag for this pair.
translated_tokens = model.generate(input_ids, max_length=60)
translated_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

print(translated_text)

# Clean up GPU memory
del model, tokenizer
torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Les données sont la nouvelle huile, mais seulement si elle est propre.


---
## Phase 1 — Data Curation with Argilla
### *HF Course Chapters 3, 5*

#### The principle: your model can never be better than your labels

This is the step most beginners skip and most senior engineers obsess over. A translation model trained on noisy, machine-generated, or inconsistent pairs will confidently learn the noise. There is no clever architecture or bigger model that fixes bad labels — garbage in, garbage out is not a cliché, it's the single most reliable predictor of whether an NLP project succeeds.

**Argilla** is a human-in-the-loop annotation tool: you load raw candidate pairs into it, and a human (you) reviews each one and marks it as correct, incorrect, or "needs editing" with a manual fix. That label schema — accept / correct / reject — is the same three-way decision every annotation pipeline in this space eventually reaches, including the six-category tagset you designed for your own Luglish corpus research. Practicing it here at small scale is direct rehearsal for that larger project.

You've already run this once and produced a small set of human-verified rows. The cells below are kept so the whole pipeline is reproducible from scratch — you don't have to re-run them if your backup CSV already exists.

In [ ]:
# Phase 1A — (Optional re-run) Pull annotations from your Argilla Space.
# Skip this cell if you already have argilla_luganda_backup.csv from a previous run.
!pip install -q argilla

import argilla as rg
from google.colab import userdata

ARGILLA_API_KEY = userdata.get('ARGILLA_API_KEY')

client = rg.Argilla(
    api_url="https://ssemulijoseph-nlp-data-curation.hf.space",
    api_key=ARGILLA_API_KEY,
)

print("Connecting to Argilla dataset...")
dataset = client.datasets(name="real_luganda_curation_v1")

print("Pulling records...")
hf_backed_up_dataset = dataset.records.to_datasets()
print(f"Total rows pulled: {len(hf_backed_up_dataset)}")

hf_backed_up_dataset.to_csv("argilla_luganda_backup.csv")
print("Backup saved: argilla_luganda_backup.csv")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.3/161.3 kB 6.4 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/argilla/_api/_token.py:83: UserWarning: 
The secrets ARGILLA_API_URL and does not exist in your Colab secrets.
  warnings.warn(f"\nThe secrets {name} and does not exist in your Colab secrets.")


Connecting to Argilla dataset...
Pulling records...
Total rows pulled: 100


Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Backup saved: argilla_luganda_backup.csv


#### Turning raw annotation exports into a clean training signal

Argilla exports every record whether or not it's been annotated yet, and stores your responses as string-encoded lists (e.g. `"['Yes']"`) rather than plain values — a common quirk of annotation-tool exports that you have to parse defensively. The function below implements exactly the three-way decision described above:

- **"Yes"** → keep the original pair as-is, tagged `human_verified`
- **"Needs Editing"** → keep your manual correction instead of the original, tagged `human_corrected`
- **"No"** → drop the row entirely; it's not worth training on a pair you've already judged wrong

Notice the `try/except` around `ast.literal_eval` — this is defensive parsing, a habit worth keeping any time you're reading semi-structured data exported by a tool you don't control. A malformed row should be skipped, not crash your whole pipeline.

In [ ]:
# Phase 1B — Extract the "golden" human-verified/corrected pairs.
import pandas as pd
import ast

def extract_golden_dataset(csv_path: str) -> pd.DataFrame:
    """Turn raw Argilla export into clean (english, luganda, source) pairs."""
    df = pd.read_csv(csv_path)
    clean_pairs = []

    for _, row in df.iterrows():
        if pd.isna(row.get('quality.responses')):
            continue
        try:
            quality_list = ast.literal_eval(row['quality.responses'])
            quality = quality_list[0] if quality_list else "No"
        except (ValueError, SyntaxError):
            continue

        english_text = row['english']

        if quality == "Yes":
            clean_pairs.append({
                "english": english_text,
                "luganda": row['luganda'],
                "source": "human_verified",
            })
        elif quality == "Needs Editing" and pd.notna(row.get('correction.responses')):
            try:
                correction_list = ast.literal_eval(row['correction.responses'])
                luganda_text = correction_list[0] if correction_list else row['luganda']
                clean_pairs.append({
                    "english": english_text,
                    "luganda": luganda_text,
                    "source": "human_corrected",
                })
            except (ValueError, SyntaxError):
                continue
        # quality == "No" -> drop (unsalvageable)

    return pd.DataFrame(clean_pairs)

golden_df = extract_golden_dataset("argilla_luganda_backup.csv")
print(f"Golden Dataset Size: {len(golden_df)} rows")
golden_df.head()


Golden Dataset Size: 7 rows


,english,luganda,source
0,Where can we get good resistant varieties that...,Tusobola kuggya wa ebika ebirungi ebigumira em...,human_verified
1,Every Monday farmers are given advice on coff...,Buli Mande abalimi bawabulwa ku nnima y'emmwanyi,human_verified
2,Farmers are given a platform to ask any agricu...,Abalimi baweebwa omwagaanya okubuuza ekibuuzo ...,human_verified
3,Farming programs aired on radio help farmers t...,Pulogulaamu z'ebyobulimi eziweerezebwa ku laad...,human_verified
4,Maize leaf rust causes the tip of a leaf to dry.,Okumyukirira kw'ebikoola bya kasooli kireetera...,human_verified


#### Why a handful of golden rows is a feature, not a failure

It's tempting to look at a small golden set and feel like the curation phase "didn't produce enough." Reframe this: a small, **trustworthy** set of rows you personally verified is far more valuable per-row than thousands of unverified scraped pairs, because you can trust every single one of its judgments completely.

The mistake would be trying to *train* a model on only this handful of rows — with so few examples, a model can simply memorize them outright rather than learning any general English→Luganda mapping, and your evaluation would then be measuring memorization, not translation ability. The professional pattern here is to split responsibilities: a small, hand-verified set becomes your **evaluation** signal (because you trust it), and a larger, less-curated public corpus becomes your **training** signal (because you need volume). We do exactly that for the rest of this notebook: `golden_df` becomes a permanent eval set, and `kambale/luganda-english-parallel-corpus` — a public dataset with thousands of pairs — supplies the training volume for Phases 2 and 4.

In [ ]:
from datasets import Dataset, load_dataset

# Convert the golden eval set into a HF Dataset object (Ch.3: Datasets library).
golden_eval_dataset = Dataset.from_pandas(golden_df[["english", "luganda"]], preserve_index=False)
print(golden_eval_dataset)

# Pull the larger public corpus for training.
print("\nDownloading public training corpus...")
full_corpus = load_dataset("kambale/luganda-english-parallel-corpus", split="train")
print(full_corpus)


Dataset({
    features: ['english', 'luganda'],
    num_rows: 7
})



README.md:   0%|          | 0.00/2.62k [00:00<?, ?B/s]

luganda-english-parallel-corpus.json:   0%|          | 0.00/4.22M [00:00<?, ?B/s]

luganda-english-parallel-corpus.jsonl:   0%|          | 0.00/3.55M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50012 [00:00<?, ? examples/s]

Dataset({
    features: ['english', 'luganda'],
    num_rows: 50012
})


---
## Phase 2 — The Baseline: a Small Seq2Seq Translation Model
### *HF Course Chapters 3, 7*

#### The engineering principle: always build the dumbest thing that could work, first

This is software engineering rule #1, and it applies to ML doubly. Before reaching for an 8-billion-parameter LLM and an expensive QLoRA fine-tune, fine-tune a **small, purpose-built translation model** — `facebook/nllb-200-distilled-600M`, an encoder-decoder model from Meta's "No Language Left Behind" project that already has some exposure to Luganda (`lug_Latn`) during its original pretraining.

Why bother, if we're going to try an LLM later anyway? Three reasons that matter in real projects, not just this exercise:

1. **It gives you a number to beat.** Without a baseline BLEU score, you have no way to know if the much more expensive LLM fine-tune in Phase 4 was actually worth the GPU-hours. "Worse but more complex" is a real failure mode in ML, and you can only catch it by comparing against something simpler.
2. **It's a data-quality canary.** If even a model purpose-built for translation, with prior exposure to your target language, struggles badly, that's a strong signal the problem is your *data* (Phase 1), not your *model choice* — and no amount of LLM sophistication fixes bad data.
3. **It's cheap to iterate on.** A 600M-parameter model trains in minutes on a free T4. An LLM QLoRA run takes much longer. You want to catch tokenization bugs, data leakage, and metric mistakes on the cheap model before you pay the expensive model's compute cost for the same mistake.

#### Why fractions, not fixed row counts, for the train/test split

A natural first instinct is to write `train_test_split(train_size=2000, test_size=200)`. The problem: that line silently assumes your corpus has at least 2,200 rows. If the public dataset is ever smaller, or you swap in your own scraped corpus later, that exact call throws a `ValueError` instead of just adapting. Using **percentages** (`test_size=0.1`) and capping the absolute count with `min(..., MAX_ROWS)` makes the split robust to dataset size while still keeping training fast on a free GPU — the kind of defensive coding that saves you from re-debugging the same split logic every time your data changes shape.

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

MODEL_CHECKPOINT = "facebook/nllb-200-distilled-600M"
SOURCE_LANG = "eng_Latn"
TARGET_LANG = "lug_Latn"
MAX_LENGTH = 128
MAX_TRAIN_ROWS = 3000   # cap so this stays fast on a free T4
MAX_EVAL_ROWS = 300

# Reuse full_corpus from Phase 1 if present, else reload.
try:
    full_corpus
except NameError:
    full_corpus = load_dataset("kambale/luganda-english-parallel-corpus", split="train")

n = len(full_corpus)
train_n = min(int(n * 0.9), MAX_TRAIN_ROWS)
eval_n = min(int(n * 0.1), MAX_EVAL_ROWS)

split = full_corpus.shuffle(seed=42).select(range(train_n + eval_n)).train_test_split(
    test_size=eval_n, seed=42
)
train_dataset = split["train"]
eval_dataset = split["test"]
print(f"Train rows: {len(train_dataset)} | Eval rows: {len(eval_dataset)}")

Using device: cuda
Train rows: 3000 | Eval rows: 300


In [ ]:
# Ensure 'evaluate' is installed.
!pip install -q evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


#### What's actually happening in the cell below, piece by piece

This is a complete, standard `Seq2SeqTrainer` loop — worth understanding line-group by line-group rather than treating as a black box, because you'll reuse this exact shape for any future encoder-decoder fine-tune:

- **`text_target=`** tells the tokenizer "these are the target-language strings, encode them as labels, not as input." Under the hood this routes through the *target* tokenizer settings (important for NLLB, which uses different language codes on each side), so you don't need a separate `with tokenizer.as_target_tokenizer():` context manager — that pattern is deprecated and `text_target` is the direct replacement.
- **`compute_metrics`** runs after every evaluation pass. Two details matter here: model outputs use `-100` as a padding/ignore marker for label positions that shouldn't count toward loss, so before decoding labels back to text we have to replace `-100` with the real pad token — otherwise the tokenizer chokes trying to decode an ID that doesn't exist in its vocabulary. `postprocess_text` then just strips whitespace so BLEU isn't penalized for trailing spaces.
- **BLEU** (via `sacrebleu`) counts overlapping word sequences (n-grams) between the model's translation and the human reference. It's an imperfect metric — it can't tell if your model got the *meaning* right with different word choices — but it's the field standard for a reason: it's automatic, reproducible, and good enough to compare two model versions against each other, which is exactly what we need it for here (Phase 2 baseline vs. Phase 4 LoRA model).
- **`fp16=torch.cuda.is_available()`** trains in 16-bit floating point instead of 32-bit whenever a GPU is present, roughly halving memory use and often speeding up training, with negligible accuracy cost for a model this size.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT, src_lang=SOURCE_LANG, tgt_lang=TARGET_LANG
)

def preprocess_function(examples):
    return tokenizer(
        examples["english"],
        text_target=examples["luganda"],
        max_length=MAX_LENGTH,
        truncation=True,
    )

print("Tokenizing...")
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_eval = eval_dataset.map(preprocess_function, batched=True, remove_columns=eval_dataset.column_names)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT).to(device)

metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [p.strip() for p in preds]
    labels = [[l.strip()] for l in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb-luganda-baseline",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=2,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting baseline training...")
trainer.train()
baseline_metrics = trainer.evaluate()
print("Baseline evaluation:", baseline_metrics)


Tokenizing...


Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Starting baseline training...


Epoch,Training Loss,Validation Loss,Bleu
1,3.241028,1.434938,16.665640
2,2.669747,1.416855,17.978677


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Bleu
2.669747,1.416855,2,17.978677


Baseline evaluation: {'eval_loss': 1.416854977607727, 'eval_bleu': 17.978676743206478}


#### Why we also eyeball translations, not just trust the BLEU number

A single aggregate BLEU score over hundreds of eval rows can hide real problems — a model can score reasonably while still producing translations that are subtly wrong or generic for certain sentence types. Reading actual predictions next to actual references, especially against your **hand-verified golden set**, catches things a single number can't: word order issues, words it refuses to translate, or systematic mistakes on a particular sentence structure. Treat this qualitative check as a required step, not an optional nicety — in production NLP work, "the metric looks fine" and "the outputs are actually good" are not the same claim, and you should always verify both.

In [ ]:
# Sanity-check the baseline against your 7-row human-verified gold set.
def translate_batch(texts, model, tokenizer, max_length=128):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
    forced_bos = tokenizer.convert_tokens_to_ids(TARGET_LANG)
    outputs = model.generate(**inputs, forced_bos_token_id=forced_bos, max_length=max_length)
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

sample_en = golden_eval_dataset["english"][:5]
sample_lg = golden_eval_dataset["luganda"][:5]
preds = translate_batch(sample_en, model, tokenizer)

for en, ref, pred in zip(sample_en, sample_lg, preds):
    print(f"EN  : {en}")
    print(f"REF : {ref}")
    print(f"PRED: {pred}")
    print("---")


EN  : Where can we get good resistant varieties that are highly marketable?
REF : Tusobola kuggya wa ebika ebirungi ebigumira embeera yonna bya ttunzi nnyo?
PRED: Wa w'oyinza okufuna ebika by'enva endiga ebirinda ebiruma ebisasulwa ennyo?
---
EN  : Every Monday  farmers are given advice on coffee growing.
REF : Buli Mande abalimi bawabulwa ku nnima y'emmwanyi
PRED: Buli lwa Bbalaza abalimi baweebwa amagezi ku ngeri y'okukulaamu kofi.
---
EN  : Farmers are given a platform to ask any agricultural related questions.
REF : Abalimi baweebwa omwagaanya okubuuza ekibuuzo kyonna ekikwata ku byobulimi n'obulunzi.
PRED: Abalimi baweebwa ekifo okwebuuza ebibuuzo ebikwata ku bulimi.
---
EN  : Farming programs aired on radio help farmers to know the good planting seasons.
REF : Pulogulaamu z'ebyobulimi eziweerezebwa ku laadiyo ziyamba balimi okumanya sizoni ennungi ez'okusimbiramu.
PRED: Ebirimu eby'obulimi ebyabanga ku leediyo biyamba abalimi okumanya ebiro by'okusiga ebirungi.
---
EN  : Maize le

#### Free the GPU before moving on

A free Colab T4 has roughly 15GB of VRAM — plenty for one model at a time, not necessarily enough for two loaded simultaneously plus a training run. Explicitly deleting a model object and calling `torch.cuda.empty_cache()` once you're done with it is the single highest-leverage habit for avoiding CUDA out-of-memory errors later in the notebook. Python's garbage collector eventually reclaims unused objects on its own, but "eventually" is not good enough when the next cell needs that memory immediately.

In [ ]:
# Free VRAM before moving to Phase 3/4 — this is the single most common
# fix for the CUDA OOM errors you'll hit later if you skip it.
import gc
del model, trainer
gc.collect()
torch.cuda.empty_cache()
print("Baseline model unloaded, VRAM freed.")


Baseline model unloaded, VRAM freed.


---
## Phase 3 — Tokenizers in Depth & the Hugging Face Hub
### *HF Course Chapters 4, 6*

#### Why look inside the tokenizer at all

Every cell so far has called a tokenizer without inspecting what it actually does to text. That's normally fine — until something goes wrong, and the bug is *inside* tokenization (a special token in the wrong place, a language code mismatch, an unexpected subword split for a word in your target language). When that happens, you need to know how to open the hood.

Most modern HF tokenizers are **subword** tokenizers: rather than one token per whole word (which would need an enormous vocabulary to cover every language) or one token per character (which would lose all word-level meaning), they break unfamiliar or rare words into smaller frequent pieces. This matters specifically for Luganda: as a lower-resource language, many Luganda words will not appear whole in NLLB's vocabulary and will get split into multiple subword pieces. Seeing this directly, on a real Luganda sentence, makes that abstract fact concrete.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")

sample = "Abalimi mu Mbarara basaba obuyambi ku by'okulima."  # Luganda agriculture sentence
encoded = tok(sample)
print("Input IDs   :", encoded["input_ids"])
print("Tokens      :", tok.convert_ids_to_tokens(encoded["input_ids"]))
print("Decoded back:", tok.decode(encoded["input_ids"]))
print("\nSpecial tokens map:", tok.special_tokens_map)
print("Vocab size  :", tok.vocab_size)


Input IDs   : [256047, 70, 4113, 517, 216, 72, 2078, 188, 245834, 11358, 234826, 90, 811, 248116, 767, 19978, 248075, 2]
Tokens      : ['eng_Latn', '▁A', 'bali', 'mi', '▁mu', '▁M', 'bar', 'ara', '▁basaba', '▁obu', 'yambi', '▁ku', '▁by', "'", 'oku', 'lima', '.', '</s>']
Decoded back: eng_Latn Abalimi mu Mbarara basaba obuyambi ku by'okulima.</s>

Special tokens map: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}
Vocab size  : 256204


#### The Hub is not just storage — it's how your work becomes reusable

Pushing a model to the Hub does three things at once: it versions your weights (so you can roll back), it gives the model a permanent, shareable identifier anyone can load with `from_pretrained("your-username/repo-name")`, and — if you attach a model card — it documents what the model is for and what it isn't good at, so someone else (including future-you) doesn't have to reverse-engineer that from the weights alone. This is the difference between a model that lives only in a Colab session that disappears in 12 hours, and a model that's a permanent, citable artifact of your work.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import os

MODEL_CHECKPOINT = "facebook/nllb-200-distilled-600M"
HF_USERNAME = "ssemulijoseph"
REPO_NAME = f"{HF_USERNAME}/nllb-luganda-baseline"

# Find the actual checkpoint directory inside the output folder
output_dir = "./nllb-luganda-baseline"
checkpoints = [os.path.join(output_dir, d) for d in os.listdir(output_dir) if d.startswith("checkpoint")]
load_path = max(checkpoints, key=os.path.getmtime) if checkpoints else output_dir

print(f"Reloading model from: {load_path}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = AutoModelForSeq2SeqLM.from_pretrained(load_path)

# Pushing to Hugging Face Hub
# IMPORTANT: Only uncomment these after updating your HF_TOKEN to 'Write' access in Secrets
print(f"Target Hub repo: {REPO_NAME}")
tokenizer.push_to_hub(REPO_NAME)
model.push_to_hub(REPO_NAME)


Reloading model from: ./nllb-luganda-baseline/checkpoint-376


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

Target Hub repo: ssemulijoseph/nllb-luganda-baseline


HfHubHTTPError: Client error '401 Unauthorized' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6a310817-71021f61354d8c95177a785e;07e714c9-49be-4df8-8eb4-e2cf47205611)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.

---
## Phase 4 — Parameter-Efficient Fine-Tuning with QLoRA
### *HF Course Chapter 11*

#### The problem QLoRA actually solves

A modern instruction-following LLM has billions of parameters. Full fine-tuning — updating *every* one of those parameters — requires storing the model weights, the gradients, and the optimizer state all at once, in 32-bit precision, in GPU memory. For a multi-billion-parameter model that's far more memory than a free Colab T4 (about 15GB) has available. QLoRA is the engineering answer to that constraint, and it's worth understanding both halves of the name separately, because they solve two different problems:

- **Q — Quantization.** Load the *base* model's weights in 4-bit precision instead of 32-bit, shrinking its memory footprint roughly 8x. The base model is then frozen — we never update these weights, so a lower-precision approximation of them is good enough.
- **LoRA — Low-Rank Adaptation.** Rather than updating the frozen base weights at all, inject small trainable "adapter" matrices alongside selected layers, and train *only those*. A 1.5B-parameter base model might end up with only a few million trainable adapter parameters — small enough to fit comfortably in the memory the quantization step just freed up, and small enough that the resulting adapter file is megabytes, not gigabytes.

Put together: quantization shrinks what you have to hold in memory, LoRA shrinks what you actually have to train. Neither idea alone gets you fine-tuning an LLM on a free GPU; together, they do.

#### Why Qwen2.5-1.5B instead of an 8-billion-parameter model

Even with QLoRA, an 8B model still has real memory and time costs — loading it, even quantized, takes a meaningful chunk of a T4's VRAM, and a single epoch takes proportionally longer. `Qwen/Qwen2.5-1.5B-Instruct` is roughly 5x smaller, already instruction-tuned (so it follows a `"Translate to Luganda: ..."` style prompt reasonably out of the box), has no gated-access approval step to wait for (unlike Llama-3), and is genuinely fast enough to iterate on multiple times in one sitting on a free T4. Once you've verified this entire pipeline runs cleanly end-to-end on the small model, swapping `MODEL_ID` to a larger model later is a one-line change — the code does not change, only your patience and VRAM headroom need to.

#### Why we frame this as instruction-following, not just translation

Rather than feeding the model raw `(english, luganda)` pairs the way the Phase 2 seq2seq model wants them, we wrap each pair into a chat-style instruction: `"Translate to Luganda: {english}"` → `{luganda}`. This matters because Qwen2.5-Instruct was pretrained to expect conversational turns, not bare sentence pairs — training it the way it was *already* shaped to be used gets you a usable model with far less data and far fewer steps than fighting its existing format would.

In [ ]:
import torch
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SFT_ROWS = 2000  # keep training short for a free T4

try:
    full_corpus
except NameError:
    full_corpus = load_dataset("kambale/luganda-english-parallel-corpus", split="train")

sft_source = full_corpus.shuffle(seed=42).select(range(min(len(full_corpus), MAX_SFT_ROWS)))

def to_chat_format(example):
    return {
        "messages": [
            {"role": "user", "content": f"Translate to Luganda: {example['english']}"},
            {"role": "assistant", "content": example["luganda"]},
        ]
    }

sft_dataset = sft_source.map(to_chat_format, remove_columns=sft_source.column_names)
print(sft_dataset[0])


#### Reading the quantization and LoRA config like an engineer, not a copy-paster

Each argument below is a deliberate choice, not boilerplate:

- **`bnb_4bit_quant_type="nf4"`** — NF4 ("NormalFloat4") is a 4-bit format specifically designed for neural network weights, which tend to cluster near zero in a roughly bell-curve distribution. It represents that distribution more accurately than a naive linear 4-bit encoding would, for the same memory cost.
- **`bnb_4bit_use_double_quant=True`** — quantizes the quantization constants themselves, squeezing out a small amount of additional memory for free.
- **`r=16` (LoRA rank)** — the dimensionality of the adapter matrices. Higher rank means more trainable capacity (and more memory/compute), lower rank means a tighter bottleneck the model must compress its learning through. 16 is a well-tested middle ground for a task this narrow (translation) on a model this size — we're not asking the model to learn an entirely new domain, just a focused input→output mapping.
- **`target_modules="all-linear"`** — attaches adapters to every linear layer rather than hand-picking specific ones, which is the simpler, more robust default unless you have a specific reason to be more surgical.
- **`model.config.use_cache = False`** — the key-value cache that speeds up *generation* actively conflicts with gradient checkpointing during *training* (both want to manage activation memory differently); disabling it here is required, not optional, whenever gradient checkpointing is on.

In [ ]:
# 4-bit quantization config — this is the "Q" in QLoRA.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False  # required for training with gradient checkpointing

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

print("Base model + LoRA config ready.")


#### Reading the training config

- **`gradient_accumulation_steps=4`** with a small `per_device_train_batch_size=4` simulates a larger effective batch size (16) without needing the memory a true batch of 16 would cost — gradients are accumulated across 4 forward/backward passes before a single optimizer step. This is the standard trick for training reasonably-sized batches on memory-constrained hardware.
- **`learning_rate=2e-4`** is deliberately higher than the `5e-5` used for the full-fine-tune baseline in Phase 2. LoRA adapters start from zero and need to learn their entire contribution from scratch in relatively few steps, so they tolerate — and need — a larger learning rate than full fine-tuning does.
- **`packing=False`** keeps each training example as its own sequence rather than concatenating multiple short examples into one packed sequence. Packing is a throughput optimization that's most valuable on very large datasets; for a dataset this size, the added bookkeeping isn't worth the complexity it introduces while you're still learning the pipeline.

In [ ]:
sft_config = SFTConfig(
    output_dir="./qwen-luganda-lora",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=20,
    save_strategy="epoch",
    save_total_limit=1,
    bf16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    report_to="none",
    max_length=256,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=sft_dataset,
    peft_config=peft_config,
)

print("Starting QLoRA fine-tuning...")
trainer.train()


#### Why pushing only the adapter is the whole point

Notice we push `trainer.model` — with `peft_config` attached, this is a small adapter wrapper, not the full 1.5B-parameter base model. The upload is typically just a few megabytes. Anyone who wants to use your fine-tuned model loads the original public Qwen2.5-1.5B-Instruct base plus your tiny adapter on top, rather than you having to host and distribute several gigabytes of (mostly unchanged) base weights. This is the practical, everyday payoff of parameter-efficient fine-tuning: cheap to train *and* cheap to share.

In [ ]:
# Push LoRA adapter weights to the Hub (Ch.6/11): tiny upload, base model stays put.
HF_USERNAME = "ssemulijoseph"   # <-- change if needed
ADAPTER_REPO = f"{HF_USERNAME}/qwen2.5-1.5b-luganda-lora"

# trainer.model.push_to_hub(ADAPTER_REPO)
# tokenizer.push_to_hub(ADAPTER_REPO)
print(f"Target adapter repo: {ADAPTER_REPO} (uncomment push_to_hub calls when ready)")


#### The moment of truth: compare against the same golden rows the baseline saw

This is why Phase 1's golden set exists. Running the exact same five hand-verified sentences through both Phase 2's baseline and this LoRA-tuned model gives you a direct, fair, side-by-side comparison — not "does this feel better," but "on identical inputs, which model's output is closer to what a human Luganda speaker actually wrote." Keep this comparison habit: any time you have two model candidates, evaluate them on exactly the same inputs, never different samples.

In [ ]:
# Quick qualitative check against the golden eval set from Phase 1.
def generate_reply(prompt, model, tokenizer, max_new_tokens=64):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
    output = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)

for en, ref in zip(golden_eval_dataset["english"][:5], golden_eval_dataset["luganda"][:5]):
    pred = generate_reply(f"Translate to Luganda: {en}", model, tokenizer)
    print(f"EN  : {en}")
    print(f"REF : {ref}")
    print(f"PRED: {pred}")
    print("---")


---
## Phase 5 — Debugging & MLOps Hygiene
### *HF Course Chapter 8*

#### Treat failure as a certainty, not an exception

On a free Colab T4, you will eventually hit a CUDA out-of-memory error or a training run that produces `NaN` losses from exploding gradients. This is not a sign you did something wrong — it's a routine, expected part of working with constrained hardware, and every NLP engineer who has ever trained on a budget GPU has hit both. The professional difference is not "avoiding it ever happens" — it's having a fast, repeatable way to diagnose it when it does, instead of panicking and restarting the runtime blind.

The three tools below are meant to become permanent habits you carry into any future training notebook, not one-off code for this project specifically.

#### Tool 1: an environment report you can paste into a bug report

The single most common reason an open-source maintainer can't help with a GitHub issue is that the reporter didn't say which exact versions of which exact libraries they were running. `environment_report()` captures that in one call, formatted as a ready-to-paste Markdown code block — the same habit, automated, that we built into Phase 0's pinned install.

In [ ]:
import torch, traceback, sys, platform
import transformers, datasets, accelerate, peft, trl

def environment_report():
    """Generate a GitHub-issue-ready environment block."""
    report = {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "accelerate": accelerate.__version__,
        "peft": peft.__version__,
        "trl": trl.__version__,
    }
    print("```")
    for k, v in report.items():
        print(f"{k}: {v}")
    print("```")
    return report

environment_report()


#### Tool 2: catching CUDA OOM with a remediation checklist attached

`safe_train_step()` wraps `trainer.train()` so that a CUDA out-of-memory error doesn't just print a wall of traceback and leave you guessing — it prints the traceback **and** an ordered list of the fixes that actually work, cheapest and least disruptive first. Notice the order is deliberate: lowering batch size and raising gradient accumulation gets you the same effective batch size for a fraction of the memory cost, so it's almost always the first thing to try before reaching for something more drastic like switching models entirely.

In [ ]:
def safe_train_step(trainer):
    """Wrap a Trainer.train() call with OOM-aware error handling and remediation hints."""
    try:
        trainer.train()
    except torch.cuda.OutOfMemoryError as e:
        print("CUDA OUT OF MEMORY caught.")
        print(traceback.format_exc())
        print("""
Remediation checklist (in order of impact):
  1. Reduce per_device_train_batch_size and raise gradient_accumulation_steps to compensate.
  2. Enable gradient_checkpointing=True in TrainingArguments/SFTConfig.
  3. Reduce max_length / max_seq_length.
  4. Call torch.cuda.empty_cache() + gc.collect() after deleting any previous model object.
  5. For QLoRA: confirm load_in_4bit=True actually applied (check model.hf_device_map).
""")
        torch.cuda.empty_cache()
    except RuntimeError as e:
        if "CUDA" in str(e):
            print("CUDA-related RuntimeError caught:")
            print(traceback.format_exc())
        else:
            raise

print("safe_train_step() defined — wrap any trainer.train() call in it, e.g.:")
print("  safe_train_step(trainer)")


#### Tool 3: catching exploding gradients before they wreck a whole run

A "gradient explosion" is when the gradient magnitude during backpropagation grows uncontrollably large, usually from a learning rate that's too high for the current model/data combination, or from a few corrupted/outlier rows in your training data. Left unchecked, it pushes weights to `NaN`, silently ruining the rest of training — you often don't notice until you check the loss curve and find it's been `NaN` for the last several hundred steps. `check_for_exploding_gradients()` scans parameter gradient norms right after a backward pass, so you catch the problem at step 5, not step 500.

In [ ]:
def check_for_exploding_gradients(model, threshold=10.0):
    """Scan current parameter gradients for norms above `threshold`."""
    exploded = []
    for name, param in model.named_parameters():
        if param.grad is not None:
            grad_norm = param.grad.data.norm(2).item()
            if grad_norm > threshold:
                exploded.append((name, round(grad_norm, 2)))
    if exploded:
        print(f"{len(exploded)} parameter(s) with gradient norm > {threshold}:")
        for name, norm in exploded[:10]:
            print(f"  {name}: {norm}")
        print("Consider: lowering learning_rate, adding max_grad_norm=1.0 to TrainingArguments,")
        print("or checking for bad/duplicate rows in the training data.")
    else:
        print(f"No exploding gradients detected (threshold={threshold}).")
    return exploded

print("check_for_exploding_gradients(model) is ready to call after a backward pass.")


#### Why a structured bug report matters more than it seems

When something genuinely is a library bug (not your code), a vague "it doesn't work" issue on GitHub gets ignored — maintainers triaging dozens of issues a day can't act on it. A report with environment details, a minimal reproduction, and the actual full traceback can often be diagnosed and fixed within the same day. This is a skill, not a formality: writing a reproducible bug report is itself a form of debugging, because the act of isolating a minimal repro frequently reveals that the "bug" was actually a mistake in your own code, which you then fix yourself before ever filing anything.

```
### Environment
(paste environment_report() output here)

### Reproduction
Minimal code that reproduces the issue (ideally <20 lines).

### Expected behavior
What you expected to happen.

### Actual behavior
Full traceback, not just the last line.
```

---
## Phase 6 — Gradio Demo & Hugging Face Spaces
### *HF Course Chapter 9*

#### A model that only you can run is not yet a finished project

Everything up to this point lives inside a Colab session that disappears when you close the tab. The last mile of real NLP engineering work is making the result usable by someone who isn't you and doesn't know Python — your Sunbird AI supervisor, a fellow student, a potential employer reviewing your portfolio. **Gradio** exists specifically to close that gap: a few lines of declarative UI code, and you have a shareable web interface, no frontend framework required.

We deliberately put both models — the Phase 2 baseline and the Phase 4 LoRA model — side by side in the same interface rather than picking a "winner" to ship alone. This turns the demo itself into evidence of the comparison you did in Phase 4's final cell: a viewer doesn't have to take your word for which model is better, they can type a sentence and see both outputs at once.

In [ ]:
import gradio as gr

def translate_with_baseline(text):
    preds = translate_batch([text], model_baseline, tokenizer_baseline)
    return preds[0]

def translate_with_lora(text):
    return generate_reply(f"Translate to Luganda: {text}", model_lora, tokenizer_lora)

with gr.Blocks(title="English -> Luganda Translator") as demo:
    gr.Markdown("# English -> Luganda Translator\nCompare a fine-tuned NLLB baseline against a QLoRA-tuned Qwen2.5 model.")
    with gr.Row():
        inp = gr.Textbox(label="English text", placeholder="Type a sentence to translate...")
    with gr.Row():
        btn = gr.Button("Translate", variant="primary")
    with gr.Row():
        out_baseline = gr.Textbox(label="NLLB Baseline (Phase 2)")
        out_lora = gr.Textbox(label="Qwen2.5 + LoRA (Phase 4)")

    btn.click(translate_with_baseline, inputs=inp, outputs=out_baseline)
    btn.click(translate_with_lora, inputs=inp, outputs=out_lora)

    gr.Examples(
        examples=[
            "Where can farmers get help with maize disease?",
            "The clinic opens at eight in the morning.",
            "Thank you for your help today.",
        ],
        inputs=inp,
    )

demo.launch(share=True, debug=False)
# NOTE: model_baseline/tokenizer_baseline and model_lora/tokenizer_lora must be in memory.
# If you freed the baseline model in Phase 2's cleanup cell, reload it from
# "./nllb-luganda-baseline" before running this cell.


#### From a temporary Colab link to a permanent public URL

`demo.launch(share=True)` gives you a temporary public link that expires after your Colab session ends — fine for a quick demo to a friend, not fine for a portfolio piece you want to link from a CV. **Hugging Face Spaces** is the permanent version of the same idea: it hosts your Gradio app continuously, on a stable URL, rebuilding automatically whenever you push an update.

1. Create a new Space at huggingface.co/new-space, SDK = **Gradio**.
2. Save the Gradio app above as `app.py`, add a `requirements.txt` listing `transformers`, `torch`, `peft`, `gradio`.
3. `git push` to the Space repo, or use `huggingface_hub.upload_folder()` directly from this notebook.
4. The Space builds automatically and gives you a permanent public URL.

In [ ]:
# Optional: push the Gradio app folder straight to a Space from Colab.
from huggingface_hub import HfApi, create_repo

SPACE_ID = "ssemulijoseph/luganda-translator-demo"  # <-- change to your username

# create_repo(SPACE_ID, repo_type="space", space_sdk="gradio", exist_ok=True)
# api = HfApi()
# api.upload_file(path_or_fileobj="app.py", path_in_repo="app.py", repo_id=SPACE_ID, repo_type="space")
print(f"Target Space: {SPACE_ID} (uncomment the create_repo/upload_file calls when app.py is ready)")


---
## Phase 7 — Closing the Loop: Model Cards & Honest Documentation
### *HF Course Chapter 10*

#### Why "it works on my machine" is not the finish line

A model is only as trustworthy as its documentation. Anyone loading your model from the Hub — including you, six months from now — needs to know what data it was trained on, what its known limitations are, and what it was never tested on. For this project specifically, the honest limitations are worth stating plainly rather than glossing over: the training corpus is general-purpose and may not cover specialized domains well; the golden eval set is small (a handful of rows), so any BLEU comparison should be read as a useful signal, not a statistically rigorous benchmark; and neither model was tested on code-switched Luglish text, which is a meaningfully different problem from clean monolingual translation. Writing this down isn't an admission of failure — it's exactly what separates a credible research artifact from an overclaimed demo.

#### Closing checklist

- [ ] Phase 2 baseline BLEU score recorded and compared against Phase 4 LoRA model on the same golden eval set
- [ ] Model card written for each pushed repo (intended use, training data, limitations)
- [ ] LoRA adapter and baseline model both have Hub repos, public or private as appropriate
- [ ] Gradio demo deployed to a Space with a working public link
- [ ] `environment_report()` output saved somewhere so this project is reproducible months from now
- [ ] A README in your own GitHub repo linking to the Hub model(s) and the Space

#### Where this connects to your other work

The data curation discipline practiced in Phase 1 — a label schema, human verification, and a "needs editing" correction loop — is the same underlying pattern as the six-category tagset (LG/EN/MX/AMB/NE/OTH) you designed for your Luglish corpus research, just applied here at a much smaller scale. Having built and debugged a working version of that pattern end-to-end, on real data, before you scale it up for your Bachelor's research, is exactly the kind of hands-on grounding that makes the larger version easier to reason about and build.
